# General

## Imports

In [ ]:
import os, glob, json, random, platform, copy, sys
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from pathlib import Path

from openpyxl import load_workbook, Workbook

import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_fscore_support,
    accuracy_score,
)
from sklearn.preprocessing import label_binarize

import torchxrayvision as xrv
import torchvision.transforms as T
import timm

d:\Anaconda\envs\TFM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Set seed

In [2]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


## Métricas

In [3]:
def f1_macro_np(preds: np.ndarray, targets: np.ndarray, n_classes: int) -> float:
    f1s = []
    for c in range(n_classes):
        tp = np.sum((preds == c) & (targets == c))
        fp = np.sum((preds == c) & (targets != c))
        fn = np.sum((preds != c) & (targets == c))
        denom = 2*tp + fp + fn
        f1 = 0.0 if denom == 0 else (2.0*tp / denom)
        f1s.append(f1)
    return float(np.mean(f1s)) if len(f1s) else 0.0

def accuracy_np(preds: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean(preds == targets))


def recall_macro_np(preds: np.ndarray, targets: np.ndarray, n_classes: int) -> float:
    recalls = []
    for c in range(n_classes):
        tp = np.sum((preds == c) & (targets == c))
        fn = np.sum((preds != c) & (targets == c))
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        recalls.append(recall)
    return float(np.mean(recalls)) if len(recalls) > 0 else 0.0


In [ ]:

def append_experiment_to_excel(cfg, summary, arquitectura: str, backbone_name: str, excel_path: str):
    row = {
        "Arquitectura":      arquitectura,
        "Backbone":          getattr(cfg, "BACKBONE_NAME", backbone_name),

        "N_WAY":             getattr(cfg, "N_WAY", None),
        "N_SHOT":      getattr(cfg, "N_SHOT", None),
        "N_QUERY":     getattr(cfg, "N_QUERY", None),

        "AUGMENT_MODE":      getattr(cfg, "AUGMENT_MODE", "online"),
        "N_AUGMENTS":        getattr(cfg, "N_AUGMENTS", 1),

        "EPOCHS":            getattr(cfg, "EPOCHS", None),
        "LR":                getattr(cfg, "LR", None),
        "BACKBONE_LR":       getattr(cfg, "BACKBONE_LR", None),
        "WEIGHT_DECAY":      getattr(cfg, "WEIGHT_DECAY", None),
        "WARMUP_EPOCHS":     getattr(cfg, "WARMUP_EPOCHS", 0),
        "SCHEDULER":         getattr(cfg, "SCHEDULER", "No"),
        "NCA_LOSS":          getattr(cfg, "NCA_LOSS", "No"),
        "FINETUNE_BACKBONE": getattr(cfg, "FINETUNE_BACKBONE", False),

        "EMB_DIM":           getattr(cfg, "EMB_DIM", None),
        "PROJ_HEAD":         getattr(cfg, "PROJ_HEAD", None),
        "DROPOUT":           getattr(cfg, "DROPOUT", None),

        "TRAIN_EPISODES":    getattr(cfg, "TRAIN_EPISODES_PER_EPOCH", None),
        "VAL_EPISODES":      getattr(cfg, "VAL_EPISODES", None),
        "N_FOLDS":           getattr(cfg, "N_FOLDS", None),

        "Val_F1_mean":       round(summary.get("val_f1_mean"), 4),
        "Val_F1_std":        round(summary.get("val_f1_std"), 4),
        "Val_Acc_mean":      round(summary.get("val_acc_mean"), 4),
        "Val_Acc_std":       round(summary.get("val_acc_std"), 4),
        "Val_ROC_AUC_mean":  round(summary.get("val_roc_mean"), 4),
        "Val_ROC_AUC_std":   round(summary.get("val_roc_std"), 4),

        "Test_F1_mean":      round(summary.get("test_f1_mean"), 4),
        "Test_F1_std":       round(summary.get("test_f1_std"), 4),
        "Test_Acc_mean":     round(summary.get("test_acc_mean"), 4),
        "Test_Acc_std":      round(summary.get("test_acc_std"), 4),
        "Test_ROC_AUC_mean": round(summary.get("test_roc_mean"), 4),
        "Test_ROC_AUC_std":  round(summary.get("test_roc_std"), 4),
    }

    COLUMNS = list(row.keys())

    if os.path.exists(excel_path):
        wb = load_workbook(excel_path)
        ws = wb.active
        existing_headers = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]
        if existing_headers != COLUMNS:
            print(f"[WARNING] Las cabeceras del Excel no coinciden con las esperadas.")
            print(f"  Esperadas : {COLUMNS}")
            print(f"  En fichero: {existing_headers}")
            print("  Se añade la fila igualmente en el orden del fichero existente.")
            for col_idx, header in enumerate(existing_headers, start=1):
                ws.cell(ws.max_row + 1, col_idx).value = row.get(header)
            wb.save(excel_path)
            print(f"[OK] Fila añadida en {excel_path}")
            return
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "Resultados"
        for col_idx, col_name in enumerate(COLUMNS, start=1):
            ws.cell(1, col_idx).value = col_name
        print(f"[INFO] Excel no existía, creado en {excel_path}")

    next_row = ws.max_row + 1
    for col_idx, col_name in enumerate(COLUMNS, start=1):
        val = row[col_name]
        if isinstance(val, float) and col_name not in ("LR", "BACKBONE_LR", "WEIGHT_DECAY"):
            val = round(val, 4)
        ws.cell(next_row, col_idx).value = val

    wb.save(excel_path)
    print(f"[OK] Experimento añadido en fila {next_row} de {excel_path}")
    print(f"     Test F1: {row['Test_F1_mean']:.4f} ± {row['Test_F1_std']:.4f}  |  "
          f"Test AUC: {row['Test_ROC_AUC_mean']:.4f}")

## Dataset

In [ ]:
def list_images_in_folder(root_split_dir: str) -> Tuple[List[str], List[int], List[str]]:
    class_names = sorted([e.name for e in os.scandir(root_split_dir) if e.is_dir()])
    if not class_names:
        raise RuntimeError(f"No se encontraron clases en {root_split_dir}")
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff")
    paths, ys = [], []
    for c in class_names:
        cdir = os.path.join(root_split_dir, c)
        for ext in exts:
            for f in glob.glob(os.path.join(cdir, ext)):
                paths.append(f)
                ys.append(class_to_idx[c])
    return paths, ys, class_names


class XRayFolderDataset(Dataset):
    def __init__(self, paths: List[str], ys: List[int], class_names: List[str],
                 base_preproc=None, cache_in_ram: bool = True, cache_after_preproc: bool = True):
        self.paths = paths
        self.ys = np.array(ys, dtype=np.int64)
        self.classes = class_names
        self.base_preproc = base_preproc
        self.cache_in_ram = cache_in_ram
        self.cache_after_preproc = cache_after_preproc
        self._cache: Dict[str, np.ndarray] = {}
        if len(self.paths) == 0:
            raise RuntimeError("Dataset vacío.")

    def __len__(self):
        return len(self.paths)

    def _load_raw(self, path: str) -> np.ndarray:
        img = cv2.imread(path, cv2.IMREAD_ANYDEPTH)
        if img is None:
            raise RuntimeError(f"No pude leer {path}")
        img = img.astype(np.float32)
        img = xrv.datasets.normalize(img, MAXVAL)
        return img[None, ...]

    def _load_img(self, path: str) -> np.ndarray:
        if self.cache_in_ram and path in self._cache:
            return self._cache[path].copy()
        img = self._load_raw(path)
        if self.base_preproc is not None and self.cache_after_preproc:
            img = self.base_preproc(img)
        if self.cache_in_ram:
            self._cache[path] = img.copy()
        return img.copy()

    def __getitem__(self, idx):
        img = self._load_img(self.paths[idx])
        return torch.from_numpy(img).float(), torch.tensor(int(self.ys[idx]), dtype=torch.long)


## Transforms

In [6]:
class TorchAugment:
    def __init__(self):
        self.tf = T.Compose([
            T.RandomApply(
                [T.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05))],
                p=0.7
            ),
        ])

    def __call__(self, img_np: np.ndarray) -> np.ndarray:
        x = torch.from_numpy(img_np).float()
        return self.tf(x).numpy()


class IdentityTransform:
    def __call__(self, x):
        return x


## Matchinghead

In [ ]:
class MatchingHead(nn.Module):
    def __init__(self, init_inv_temp: float = 3.0, metric: str = "cosine"):
        super().__init__()
        self.logit_scale = nn.Parameter(
            torch.tensor(float(np.log(init_inv_temp)), dtype=torch.float32)
        )
        self.metric = metric

    def forward(self, emb_support, emb_query, y_query, n_way, n_shot):
        inv_temp = self.logit_scale.exp().clamp(0.1, 50.0)

        y_support = torch.arange(n_way, device=emb_support.device).repeat_interleave(n_shot)

        if self.metric == "cosine":
            sim = (F.normalize(emb_query, dim=1) @ F.normalize(emb_support, dim=1).t()) * inv_temp
        else:
            sim = -((emb_query[:, None, :] - emb_support[None, :, :]) ** 2).sum(-1) * inv_temp

        attn = sim.softmax(dim=1)

        logits = torch.zeros(attn.size(0), n_way, device=attn.device)
        logits.scatter_add_(1, y_support[None, :].expand(attn.size(0), -1), attn)

        loss = F.cross_entropy((logits + 1e-12).log(), y_query)

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        y_np  = y_query.detach().cpu().numpy()
        acc   = accuracy_np(preds, y_np)
        rec   = recall_macro_np(preds, y_np, n_way)
        f1    = f1_macro_np(preds, y_np, n_way)

        return (
            loss,
            torch.tensor(acc, device=logits.device),
            torch.tensor(rec, device=logits.device),
            torch.tensor(f1,  device=logits.device),
            logits,
        )

## Episodic sampler dataset

In [8]:
class EpisodicFewShotDataset(Dataset):
    def __init__(self, base_ds, n_way: int, n_shot: int, n_query: int,
                 n_episodes: int, transform_support, transform_query, seed: int = 123):
        self.ds        = base_ds
        self.n_way     = n_way
        self.n_shot    = n_shot
        self.n_query   = n_query
        self.n_episodes = n_episodes
        self.rng       = np.random.default_rng(seed)
        self.ts        = transform_support
        self.tq        = transform_query

        self.class_to_indices = defaultdict(list)
        for i in range(len(self.ds)):
            self.class_to_indices[int(self.ds.ys[i])].append(i)
        self.class_to_indices = {k: np.array(v, dtype=np.int64)
                                  for k, v in self.class_to_indices.items()}
        self.classes = sorted(self.class_to_indices)
        assert len(self.classes) >= self.n_way, f"No hay suficientes clases para n_way={n_way}"

    def __len__(self):
        return self.n_episodes

    def __getitem__(self, idx):
        classes = self.rng.choice(self.classes, size=self.n_way, replace=False).tolist()
        S_imgs, Q_imgs, Q_labels = [], [], []
        need = self.n_shot + self.n_query

        for epi_c, real_c in enumerate(classes):
            ids     = self.class_to_indices[real_c]
            picks   = self.rng.choice(ids, size=need, replace=len(ids) < need)
            for j in picks[:self.n_shot]:
                img = self.ds._load_img(self.ds.paths[int(j)])
                if self.ts is not None: img = self.ts(img)
                S_imgs.append(torch.from_numpy(img).float())
            for j in picks[self.n_shot:]:
                img = self.ds._load_img(self.ds.paths[int(j)])
                if self.tq is not None: img = self.tq(img)
                Q_imgs.append(torch.from_numpy(img).float())
                Q_labels.append(epi_c)

        return {
            "support": torch.stack(S_imgs, 0),
            "query":   torch.stack(Q_imgs, 0),
            "yq":      torch.tensor(Q_labels, dtype=torch.long),
        }


## Train / Eval

In [ ]:
@torch.no_grad()
def eval_episodic(model, head, loader, n_way, n_shot, desc="Val"):
    model.eval()
    head.eval()
    accs, recs, f1s = [], [], []
    for batch in tqdm(loader, desc=desc, leave=False, dynamic_ncols=True):
        S  = batch["support"].squeeze(0).to(device)
        Q  = batch["query"].squeeze(0).to(device)
        yq = batch["yq"].squeeze(0).to(device)
        embS = model(S)
        embQ = model(Q)
        _, acc, rec, f1, _ = head(embS, embQ, yq, n_way=n_way, n_shot=n_shot)
        accs.append(float(acc))
        recs.append(float(rec))
        f1s.append(float(f1))
    return float(np.mean(accs)), float(np.mean(recs)), float(np.mean(f1s))


@torch.no_grad()
def eval_episodic_full(model, head, loader, n_way, n_shot, class_names, out_dir, tag="val"):
    os.makedirs(out_dir, exist_ok=True)

    model.eval()
    head.eval()

    accs, recs, f1s, losses = [], [], [], []

    all_y_true = []
    all_y_pred = []
    all_probs = []

    for batch in tqdm(loader, desc=f"{tag}", leave=False, dynamic_ncols=True):
        S = batch["support"].squeeze(0).to(device)
        Q = batch["query"].squeeze(0).to(device)
        yq = batch["yq"].squeeze(0).to(device)

        embS = model(S)
        embQ = model(Q)

        loss, acc, rec, f1, logits = head(embS, embQ, yq, n_way=cfg.N_WAY, n_shot=cfg.N_SHOT)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        recs.append(float(rec.item()))
        f1s.append(float(f1.item()))

        all_y_true.append(yq.detach().cpu().numpy())
        all_y_pred.append(preds.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(all_y_true)
    y_pred = np.concatenate(all_y_pred)
    y_prob = np.concatenate(all_probs)

    n_classes = y_prob.shape[1]

    loss_mean = float(np.mean(losses))
    acc_mean = float(np.mean(accs))
    rec_mean = float(np.mean(recs))
    f1_mean = float(np.mean(f1s))

    acc_global = float(accuracy_score(y_true, y_pred))
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

    fig_cm, ax_cm = plt.subplots(figsize=(7, 6))
    im = ax_cm.imshow(cm, interpolation="nearest", cmap="Blues")
    ax_cm.figure.colorbar(im, ax=ax_cm)

    tick_labels = class_names[:n_classes] if class_names is not None else [str(i) for i in range(n_classes)]

    ax_cm.set(
        xticks=np.arange(n_classes),
        yticks=np.arange(n_classes),
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        xlabel="Predicción",
        ylabel="Real",
        title=f"Matriz de confusión ({tag})",
    )
    plt.setp(ax_cm.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = cm.max() / 2.0 if cm.size > 0 else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax_cm.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    fig_cm.tight_layout()
    cm_path = os.path.join(out_dir, f"cm_{tag}.png")
    fig_cm.savefig(cm_path, dpi=200, bbox_inches="tight")
    plt.close(fig_cm)

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(n_classes),
        target_names=tick_labels,
        zero_division=0,
        output_dict=True
    )

    y_true_bin = label_binarize(y_true, classes=np.arange(n_way))

    fpr_dict = {}
    tpr_dict = {}
    aucs_per_class = {}

    fig_roc, ax_roc = plt.subplots(figsize=(7, 6))

    for c in range(n_classes):
        y_c = y_true_bin[:, c]

        if y_c.sum() == 0 or y_c.sum() == len(y_c):
            continue

        fpr_c, tpr_c, _ = roc_curve(y_c, y_prob[:, c])
        auc_c = auc(fpr_c, tpr_c)

        fpr_dict[c] = fpr_c
        tpr_dict[c] = tpr_c
        aucs_per_class[c] = float(auc_c)

        label = tick_labels[c] if c < len(tick_labels) else str(c)
        ax_roc.plot(
            fpr_c,
            tpr_c,
            lw=1.5,
            alpha=0.8,
            label=f"{label} (AUC={auc_c:.2f})"
        )

    ax_roc.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.05])
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title(f"ROC one-vs-rest ({tag})")

    if len(fpr_dict) > 0:
        ax_roc.legend(loc="lower right", fontsize=8)

    fig_roc.tight_layout()
    roc_path = os.path.join(out_dir, f"roc_{tag}.png")
    fig_roc.savefig(roc_path, dpi=200, bbox_inches="tight")
    plt.close(fig_roc)

    try:
        roc_auc_macro = float(
            roc_auc_score(
                y_true_bin,
                y_prob,
                average="macro",
                multi_class="ovr"
            )
        )
    except ValueError:
        roc_auc_macro = None

    metrics = {
        "tag": tag,
        "loss_mean_episode": loss_mean,
        "acc_mean_episode": acc_mean,
        "rec_mean_episode": rec_mean,
        "f1_mean_episode": f1_mean,
        "acc_global": acc_global,
        "precision_macro": float(prec_macro),
        "recall_macro": float(rec_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(prec_weighted),
        "recall_weighted": float(rec_weighted),
        "f1_weighted": float(f1_weighted),
        "roc_auc_macro_ovr": roc_auc_macro,
        "aucs_per_class": aucs_per_class,
        "confusion_matrix": cm.tolist(),
        "classification_report": report,
        "n_samples": int(len(y_true)),
        "n_classes": int(n_classes),
    }

    json_path = os.path.join(out_dir, f"metrics_{tag}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"\n[{tag}]")
    print(f"  Loss episodio media: {loss_mean:.4f}")
    print(f"  Acc episodio media : {acc_mean:.4f}")
    print(f"  Recall episodio    : {rec_mean:.4f}")
    print(f"  F1 episodio        : {f1_mean:.4f}")
    print(f"  Acc global         : {acc_global:.4f}")
    print(f"  F1 macro           : {f1_macro:.4f}")
    if roc_auc_macro is not None:
        print(f"  ROC AUC macro OVR  : {roc_auc_macro:.4f}")

    return metrics

In [ ]:
def train_protonet(train_ds, val_ds, out_dir: str, build_encoder_fn):
    os.makedirs(out_dir, exist_ok=True)
    print(f"[Few-shot config] N_WAY={cfg.N_WAY} | N_SHOT={cfg.N_SHOT} | N_QUERY={cfg.N_QUERY}")


    train_epi = EpisodicFewShotDataset(
        train_ds, cfg.N_WAY, cfg.N_SHOT, cfg.N_QUERY,
        n_episodes=cfg.TRAIN_EPISODES_PER_EPOCH * cfg.EPOCHS,
        transform_support=train_transform, transform_query=eval_transform,
        seed=cfg.SEED)
    val_epi = EpisodicFewShotDataset(
        val_ds, cfg.N_WAY, cfg.N_SHOT, cfg.N_QUERY,
        n_episodes=cfg.VAL_EPISODES,
        transform_support=eval_transform, transform_query=eval_transform,
        seed=cfg.SEED + 1)

    pin = (device.type == "cuda")
    train_loader = DataLoader(train_epi, batch_size=1, shuffle=True,
                              num_workers=cfg.NUM_WORKERS, pin_memory=pin,
                              persistent_workers=(cfg.NUM_WORKERS > 0),
                              prefetch_factor=2 if cfg.NUM_WORKERS > 0 else None)
    val_loader = DataLoader(val_epi, batch_size=1, shuffle=False,
                            num_workers=cfg.NUM_WORKERS, pin_memory=pin)

    model = build_encoder_fn().to(device)
    head  = MatchingHead(cfg.TEMPERATURE_INIT_INV, metric="cosine").to(device)

    if cfg.FREEZE_ENCODER:
        params = list(head.parameters()) + [p for p in model.proj.parameters() if p.requires_grad]
    else:
        params = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())

    opt    = torch.optim.AdamW(params, lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best   = {"f1": -1.0, "model": None, "head": None}

    history = {"train_loss": [], "train_acc": [], "train_rec": [], "train_f1": [],
               "val_acc":   [], "val_rec":   [], "val_f1":   []}

    it = iter(train_loader)
    for ep in range(cfg.EPOCHS):
        model.train()
        head.train()
        losses, accs, recs, f1s = [], [], [], []
        pbar = tqdm(range(cfg.TRAIN_EPISODES_PER_EPOCH),
                    desc=f"Epoch {ep+1}/{cfg.EPOCHS}", dynamic_ncols=True)
        for _ in pbar:
            batch = next(it)
            S  = batch["support"].squeeze(0).to(device)
            Q  = batch["query"].squeeze(0).to(device)
            yq = batch["yq"].squeeze(0).to(device)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                embS = model(S)
                embQ = model(Q)
                loss, acc, rec, f1, _ = head(embS, embQ, yq,
                                             n_way=cfg.N_WAY, n_shot=cfg.N_SHOT)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss))
            accs.append(float(acc))
            recs.append(float(rec))
            f1s.append(float(f1))
            pbar.set_postfix(loss=f"{np.mean(losses):.3f}",
                             acc=f"{np.mean(accs):.3f}",
                             rec=f"{np.mean(recs):.3f}",
                             f1=f"{np.mean(f1s):.3f}",
                             invT=f"{head.logit_scale.exp().item():.2f}")

        val_acc, val_rec, val_f1 = eval_episodic(model, head, val_loader,
                                          cfg.N_WAY, cfg.N_SHOT, desc="Val")
        scheduler.step()

        tl = float(np.mean(losses))
        ta = float(np.mean(accs))
        tr = float(np.mean(recs))
        tf = float(np.mean(f1s))
        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["train_rec"].append(tr)
        history["train_f1"].append(tf)
        history["val_acc"].append(val_acc)
        history["val_rec"].append(val_rec)
        history["val_f1"].append(val_f1)
        print(f"[Epoch {ep+1}/{cfg.EPOCHS}] loss={tl:.4f} "
              f"train acc={ta:.4f} rec={tr:.4f} f1={tf:.4f} | "
              f"val acc={val_acc:.4f} rec={val_rec:.4f} f1={val_f1:.4f}")

        if val_f1 > best["f1"]:
            best["f1"]    = val_f1
            best["model"] = copy.deepcopy(model.state_dict())
            best["head"]  = copy.deepcopy(head.state_dict())
            torch.save(best["model"], os.path.join(out_dir, "best_model.pt"))
            torch.save(best["head"],  os.path.join(out_dir, "best_head.pt"))

    epochs = list(range(1, cfg.EPOCHS + 1))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(epochs, history["train_loss"], label="train loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    for ax, key, title in zip(axes[1:],
                               [("train_acc","val_acc","train_rec","val_rec"),
                                ("train_f1", "val_f1")],
                               ["Accuracy & Recall", "F1"]):
        if len(key) == 4:
            ax.plot(epochs, history[key[0]], label="train acc")
            ax.plot(epochs, history[key[1]], "--", label="val acc")
            ax.plot(epochs, history[key[2]], label="train rec")
            ax.plot(epochs, history[key[3]], "--", label="val rec")
        else:
            ax.plot(epochs, history[key[0]], label="train f1")
            ax.plot(epochs, history[key[1]], "--", label="val f1")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()

    fig.tight_layout()
    curves_path = os.path.join(out_dir, "training_curves.png")
    fig.savefig(curves_path, dpi=120)
    plt.close(fig)
    print(f"  → {curves_path}")

    model.load_state_dict(best["model"])
    head.load_state_dict(best["head"])
    with open(os.path.join(out_dir, "train_summary.json"), "w") as fh:
        json.dump({"best_val_f1": best["f1"], "history": history}, fh, indent=2)
    return model, head


In [ ]:
def make_base_splits_from_folders(base_preproc=None):
    tr_paths, tr_y, train_class_names = list_images_in_folder(os.path.join(cfg.DATA_ROOT, "train"))
    te_paths, te_y, test_class_names  = list_images_in_folder(os.path.join(cfg.DATA_ROOT, "test"))
    kw = dict(base_preproc=base_preproc, cache_in_ram=True, cache_after_preproc=True)
    train_ds = XRayFolderDataset(tr_paths, tr_y, train_class_names, **kw)
    test_ds  = XRayFolderDataset(te_paths, te_y, test_class_names,  **kw)
    print(f"Train: {len(train_ds)} imgs, {len(train_class_names)} clases")
    print(f"Test:  {len(test_ds)}  imgs, {len(test_class_names)}  clases")
    return train_ds, test_ds


class ConcatSimple(Dataset):
    def __init__(self, a, b):
        self.paths   = a.paths + b.paths
        self.ys      = np.concatenate([a.ys, b.ys], axis=0)
        self.classes = a.classes
        self._base   = a
    def __len__(self):      return len(self.paths)
    def _load_img(self, p): return self._base._load_img(p)
    def __getitem__(self, idx):
        img = self._load_img(self.paths[idx])
        return torch.from_numpy(img).float(), torch.tensor(int(self.ys[idx]), dtype=torch.long)


class SubsetSimple(Dataset):
    def __init__(self, base, indices):
        self.paths   = [base.paths[i] for i in indices]
        self.ys      = np.array([base.ys[i] for i in indices], dtype=np.int64)
        self.classes = base.classes
        self._base   = base
    def __len__(self):      return len(self.paths)
    def _load_img(self, p): return self._base._load_img(p)
    def __getitem__(self, idx):
        img = self._load_img(self.paths[idx])
        return torch.from_numpy(img).float(), torch.tensor(int(self.ys[idx]), dtype=torch.long)
    
def remap_labels(ds):
    unique = sorted(np.unique(ds.ys))
    mapping = {old: new for new, old in enumerate(unique)}
    ds.ys = np.array([mapping[y] for y in ds.ys], dtype=np.int64)
    ds.classes = [ds.classes[i] for i in unique]
    return ds


def class_kfold_indices(class_names: list, n_splits: int, seed: int):
    rng = np.random.default_rng(seed)
    classes = np.array(class_names)
    classes = rng.permutation(classes)
    folds = np.array_split(classes, n_splits)

    for fold_idx in range(n_splits):
        val_classes   = folds[fold_idx].tolist()
        train_classes = [c for i, f in enumerate(folds) if i != fold_idx for c in f]
        yield fold_idx + 1, train_classes, val_classes


## Main

In [ ]:
def run(build_encoder_fn, base_preproc=None):
    set_seed(cfg.SEED)
    train_ds, test_ds = make_base_splits_from_folders(base_preproc)
    train_class_names = train_ds.classes
    test_class_names  = test_ds.classes
    pin = (device.type == "cuda")

    if cfg.USE_CV:
        print(f"[CV] {len(train_class_names)} clases meta_train | {cfg.N_FOLDS} folds de clases")
        cv_results = []

        for fold, tr_classes, va_classes in class_kfold_indices(train_class_names, cfg.N_FOLDS, cfg.SEED):
            print(f"\n===== Fold {fold}/{cfg.N_FOLDS} =====")
            print(f"  Train clases ({len(tr_classes)}): {tr_classes}")
            print(f"  Val   clases ({len(va_classes)}): {va_classes}")
            set_seed(cfg.SEED + fold)

            tr_idx = np.where(np.isin(np.array(train_ds.classes)[train_ds.ys], tr_classes))[0]
            va_idx = np.where(np.isin(np.array(train_ds.classes)[train_ds.ys], va_classes))[0]
            tr_fold = SubsetSimple(train_ds, tr_idx)
            va_fold = SubsetSimple(train_ds, va_idx)

            tr_fold = remap_labels(tr_fold)
            va_fold = remap_labels(va_fold)

            fold_out = os.path.join(cfg.OUT_DIR, f"cv_fold_{fold}")
            os.makedirs(fold_out, exist_ok=True)

            model, head = train_protonet(tr_fold, va_fold, fold_out, build_encoder_fn)

            val_m, test_m = None, None

            for tag, ds_eval, cnames, seed_off in [
                ("val",  va_fold, va_classes,       999),
                ("test", test_ds, test_class_names, 1999),
            ]:
                epi = EpisodicFewShotDataset(
                    ds_eval, cfg.N_WAY, cfg.N_SHOT, cfg.N_QUERY,
                    n_episodes=cfg.VAL_EPISODES,
                    transform_support=eval_transform, transform_query=eval_transform,
                    seed=seed_off + fold,
                )
                loader = DataLoader(epi, batch_size=1, shuffle=False,
                                    num_workers=cfg.NUM_WORKERS, pin_memory=pin)
                m = eval_episodic_full(model, head, loader, cfg.N_WAY, cfg.N_SHOT,
                                       cnames, fold_out, tag=tag)
                if tag == "val": val_m = m
                else: test_m = m

            fold_result = {
                "fold": fold,
                "val_acc":  val_m["acc_mean_episode"],
                "val_rec":  val_m["rec_mean_episode"],
                "val_f1":   val_m["f1_mean_episode"],
                "val_roc":  val_m["roc_auc_macro_ovr"],
                "test_acc": test_m["acc_mean_episode"],
                "test_rec": test_m["rec_mean_episode"],
                "test_f1":  test_m["f1_mean_episode"],
                "test_roc": test_m["roc_auc_macro_ovr"],
                "val_loss":        val_m["loss_mean_episode"],
                "val_acc_global":  val_m["acc_global"],
                "val_f1_macro":    val_m["f1_macro"],
                "test_loss":       test_m["loss_mean_episode"],
                "test_acc_global": test_m["acc_global"],
                "test_f1_macro":   test_m["f1_macro"],
            }
            cv_results.append(fold_result)

            with open(os.path.join(fold_out, "results.json"), "w", encoding="utf-8") as fh:
                json.dump(fold_result, fh, indent=2, ensure_ascii=False)

        summary = {}
        for split in ("val", "test"):
            for metric in ("acc", "rec", "f1", "roc"):
                key  = f"{split}_{metric}"
                vals = [r[key] for r in cv_results if r[key] is not None]
                summary[f"{key}_mean"] = float(np.mean(vals)) if vals else None
                summary[f"{key}_std"]  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

        with open(os.path.join(cfg.OUT_DIR, "cv_summary.json"), "w", encoding="utf-8") as fh:
            json.dump(summary, fh, indent=2, ensure_ascii=False)

        print("\n=== CV SUMMARY ===")
        for split in ("val", "test"):
            row = "  ".join(
                f"{m}={summary[f'{split}_{m}_mean']:.4f}±{summary[f'{split}_{m}_std']:.4f}"
                if summary[f"{split}_{m}_mean"] is not None else f"{m}=None"
                for m in ("acc", "rec", "f1", "roc")
            )
            print(f"  [{split.upper()}] {row}")

        folds_x = [r["fold"] for r in cv_results]
        fig_cv, axes_cv = plt.subplots(1, 4, figsize=(16, 4))
        for ax, m in zip(axes_cv, ("acc", "rec", "f1", "roc")):
            ax.bar([f - 0.18 for f in folds_x], [r[f"val_{m}"]  or 0.0 for r in cv_results], width=0.35, label="val",  alpha=0.8)
            ax.bar([f + 0.18 for f in folds_x], [r[f"test_{m}"] or 0.0 for r in cv_results], width=0.35, label="test", alpha=0.8)
            ax.set_title(m.upper())
            ax.set_xlabel("Fold")
            ax.set_ylim(0, 1)
            ax.set_xticks(folds_x)
            ax.legend()
        fig_cv.suptitle("CV metrics per fold", fontsize=13)
        fig_cv.tight_layout()
        fig_cv.savefig(os.path.join(cfg.OUT_DIR, "cv_fold_metrics.png"), dpi=120)
        plt.close(fig_cv)

        fig_curves, axes_curves = plt.subplots(1, 3, figsize=(18, 5))
        colors   = plt.cm.tab10(np.linspace(0, 1, cfg.N_FOLDS))
        epochs_x = list(range(1, cfg.EPOCHS + 1))
        for fold_idx, fold_result in enumerate(cv_results):
            fold_num = fold_result["fold"]
            color    = colors[fold_idx]
            with open(os.path.join(cfg.OUT_DIR, f"cv_fold_{fold_num}", "train_summary.json"), "r") as f:
                fold_history = json.load(f)["history"]
            axes_curves[0].plot(epochs_x, fold_history["train_loss"], color=color, label=f"Fold {fold_num}")
            axes_curves[0].set_title("Train Loss")
            axes_curves[0].legend()
            axes_curves[1].plot(epochs_x, fold_history["train_acc"], color=color, linestyle="-",  label=f"Train F{fold_num}")
            axes_curves[1].plot(epochs_x, fold_history["val_acc"],   color=color, linestyle="--", label=f"Val F{fold_num}")
            axes_curves[1].set_title("Accuracy")
            axes_curves[1].legend(fontsize=7)
            axes_curves[2].plot(epochs_x, fold_history["train_f1"],  color=color, linestyle="-",  label=f"Train F{fold_num}")
            axes_curves[2].plot(epochs_x, fold_history["val_f1"],    color=color, linestyle="--", label=f"Val F{fold_num}")
            axes_curves[2].set_title("F1")
            axes_curves[2].legend(fontsize=7)
        fig_curves.suptitle("Curvas de aprendizaje por fold", fontsize=13)
        fig_curves.tight_layout()
        fig_curves.savefig(os.path.join(cfg.OUT_DIR, "all_folds_curves.png"), dpi=120)
        plt.close(fig_curves)

        append_experiment_to_excel(
            cfg=cfg, summary=summary,
            arquitectura="Matching", backbone_name=cfg.BACKBONE_NAME,
            excel_path="E:/TFM/Nuevos_resultados.xlsx"
        )

    else:
        model, head = train_protonet(train_ds, train_ds, cfg.OUT_DIR, build_encoder_fn)
        test_epi = EpisodicFewShotDataset(
            test_ds, cfg.N_WAY, cfg.N_SHOT, cfg.N_QUERY,
            n_episodes=cfg.VAL_EPISODES,
            transform_support=eval_transform, transform_query=eval_transform,
            seed=2024,
        )
        test_loader = DataLoader(test_epi, batch_size=1, shuffle=False,
                                 num_workers=cfg.NUM_WORKERS, pin_memory=pin)
        test_m = eval_episodic_full(model, head, test_loader, cfg.N_WAY, cfg.N_SHOT,
                                    test_class_names, cfg.OUT_DIR, tag="test")
        print(f"\n=== TEST SUMMARY ===")
        print(f"acc={test_m['acc_mean_episode']:.4f} f1={test_m['f1_mean_episode']:.4f} roc={test_m['roc_auc_macro_ovr']}")

# Backbones

## Dense net

### Config

In [ ]:
@dataclass
class CFG:
    DATA_ROOT     = "E:/TFM/Dataset_fewshot"
    OUT_DIR       = "E:/TFM/Nuevos_modelos/Outputs_matching_xrv_5way_5shot_20query"
    BACKBONE_NAME = "DenseNet121-XRV"
    IMG_SIZE   = 224
    MAXVAL     = 65535.0
    N_WAY   = 5
    N_SHOT  = 5
    N_QUERY = 20
    TRAIN_EPISODES_PER_EPOCH = 100
    VAL_EPISODES            = 200
    EPOCHS              = 15
    LR                  = 1e-4
    WEIGHT_DECAY        = 1e-4
    TEMPERATURE_INIT_INV = 3.0
    SCHEDULER = "cosine"
    XRV_WEIGHTS    = "densenet121-res224-all"
    FREEZE_ENCODER = True
    EMB_DIM        = 128
    USE_CV                = True
    N_FOLDS               = 4
    CV_OVER_TRAIN_PLUS_VAL = False
    SEED        = 42
    NUM_WORKERS = 0 if platform.system().lower().startswith("win") else 4
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

cfg    = CFG()
MAXVAL = cfg.MAXVAL
os.makedirs(cfg.OUT_DIR, exist_ok=True)
device = torch.device(cfg.DEVICE)
print("Device:", device)

base_preproc_xrv = T.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(cfg.IMG_SIZE),
])
train_transform = TorchAugment()
eval_transform  = IdentityTransform()


Device: cuda


### Encoder

In [ ]:
class XRVProtoEncoder(nn.Module):
    def __init__(self, weights: str, emb_dim: int = 128, freeze: bool = True):
        super().__init__()
        self.backbone = xrv.models.DenseNet(weights=weights)
        self.proj = nn.Sequential(
            nn.Linear(1024, emb_dim),
        )
        self.emb_dim = emb_dim
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = F.adaptive_avg_pool2d(self.backbone.features(x), 1).flatten(1)
        return F.normalize(self.proj(feats), dim=1)


### Run

In [15]:
def build_xrv():
    return XRVProtoEncoder(cfg.XRV_WEIGHTS, emb_dim=cfg.EMB_DIM, freeze=cfg.FREEZE_ENCODER)

run(build_xrv, base_preproc=base_preproc_xrv)

Train: 2000 imgs, 20 clases
Test:  435  imgs, 5  clases
[CV] 20 clases meta_train | 4 folds de clases

===== Fold 1/4 =====
  Train clases (15): [np.str_('gynecomastia'), np.str_('cardiomegaly'), np.str_('vertebral_anterior_compression'), np.str_('apical_pleural_thickening'), np.str_('alveolar_pattern'), np.str_('nodule'), np.str_('callus_rib_fracture'), np.str_('hemidiaphragm_elevation'), np.str_('vascular_hilar_enlargement'), np.str_('aortic_elongation'), np.str_('calcified_granuloma'), np.str_('scoliosis'), np.str_('aortic_atheromatosis'), np.str_('infiltrates'), np.str_('diaphragmatic_eventration')]
  Val   clases (5): ['laminar_atelectasis', 'fibrotic_band', 'interstitial_pattern', 'costophrenic_angle_blunting', 'hiatal_hernia']
[Few-shot config] N_WAY=5 | N_SHOT=5 | N_QUERY=20


Epoch 1/15: 100%|██████████| 100/100 [06:20<00:00,  3.81s/it, acc=0.353, f1=0.334, invT=3.01, loss=1.477, rec=0.353]


[Epoch 1/15] loss=1.4771 train acc=0.3535 rec=0.3535 f1=0.3339 | val acc=0.4034 rec=0.4034 f1=0.3902


Epoch 2/15: 100%|██████████| 100/100 [00:31<00:00,  3.17it/s, acc=0.374, f1=0.351, invT=3.02, loss=1.456, rec=0.374]


[Epoch 2/15] loss=1.4561 train acc=0.3738 rec=0.3738 f1=0.3513 | val acc=0.3956 rec=0.3956 f1=0.3741


...

[Epoch 14/15] loss=1.3783 train acc=0.4189 rec=0.4189 f1=0.4020 | val acc=0.3573 rec=0.3573 f1=0.3395


Epoch 15/15: 100%|██████████| 100/100 [00:33<00:00,  2.96it/s, acc=0.442, f1=0.427, invT=3.13, loss=1.345, rec=0.442]


[Epoch 15/15] loss=1.3451 train acc=0.4416 rec=0.4416 f1=0.4266 | val acc=0.3535 rec=0.3535 f1=0.3372
  → E:/TFM/Nuevos_modelos/Outputs_matching_xrv_5way_5shot_20query\cv_fold_4\training_curves.png



[val]
  Loss episodio media: 1.4779
  Acc episodio media : 0.3554
  Recall episodio    : 0.3554
  F1 episodio        : 0.3397
  Acc global         : 0.3554
  F1 macro           : 0.3554
  ROC AUC macro OVR  : 0.6778



[test]
  Loss episodio media: 1.3796
  Acc episodio media : 0.4278
  Recall episodio    : 0.4278
  F1 episodio        : 0.4048
  Acc global         : 0.4278
  F1 macro           : 0.4278
  ROC AUC macro OVR  : 0.7377

=== CV SUMMARY ===
  [VAL] acc=0.3734±0.0647  rec=0.3734±0.0647  f1=0.3547±0.0579  roc=0.6858±0.0521
  [TEST] acc=0.4324±0.0032  rec=0.4324±0.0032  f1=0.4113±0.0063  roc=0.7399±0.0028
[OK] Experimento añadido en fila 48 de E:/TFM/Nuevos_resultados.xlsx
     Test F1: 0.4113 ± 0.0063  |  Test AUC: 0.7399


## ResNet18

### Config

In [13]:
@dataclass
class CFG:
    DATA_ROOT     = "E:/TFM/Dataset_fewshot"
    OUT_DIR       = "E:/TFM/Nuevos_modelos/Outputs_matching_resnet18_5way_20shot"
    BACKBONE_NAME = "ResNet18-ImageNet"
    IMG_SIZE      = 224
    MAXVAL        = 65535.0
    N_WAY         = 5
    N_SHOT        = 20
    N_QUERY       = 10
    TRAIN_EPISODES_PER_EPOCH = 100
    VAL_EPISODES  = 200
    EPOCHS        = 15
    LR            = 1e-4
    WEIGHT_DECAY  = 1e-4
    TEMPERATURE_INIT_INV = 3.0
    FREEZE_ENCODER = True
    EMB_DIM       = 128
    USE_CV        = True
    N_FOLDS       = 4
    SEED          = 42
    NUM_WORKERS   = 0 if platform.system().lower().startswith("win") else 4
    DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

cfg    = CFG()
MAXVAL = cfg.MAXVAL
os.makedirs(cfg.OUT_DIR, exist_ok=True)
device = torch.device(cfg.DEVICE)
print("Device:", device)

base_preproc_xrv = T.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(cfg.IMG_SIZE),
])
train_transform = TorchAugment()
eval_transform  = IdentityTransform()


Device: cuda


### Encoder

In [14]:
import torchvision.models as tvm

class ResNet18ProtoEncoder(nn.Module):
    """ResNet18 (ImageNet pretrained) + projection head."""
    def __init__(self, emb_dim: int = 256, freeze: bool = True):
        super().__init__()
        backbone = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        # Quitamos la capa de clasificación final
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # feat_dim = 512
        self.proj = nn.Linear(512, emb_dim)
        self.emb_dim = emb_dim
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ResNet18 espera 3 canales
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        feats = self.backbone(x).flatten(1)  # [B, 512]
        return F.normalize(self.proj(feats), dim=1)

### Run

In [15]:
def build_resnet18():
    return ResNet18ProtoEncoder(emb_dim=cfg.EMB_DIM, freeze=cfg.FREEZE_ENCODER)

In [16]:
run(build_resnet18, base_preproc=base_preproc_xrv)

Train: 2000 imgs, 20 clases
Test:  435  imgs, 5  clases
[CV] 20 clases meta_train | 4 folds de clases

===== Fold 1/4 =====
  Train clases (15): [np.str_('gynecomastia'), np.str_('cardiomegaly'), np.str_('vertebral_anterior_compression'), np.str_('apical_pleural_thickening'), np.str_('alveolar_pattern'), np.str_('nodule'), np.str_('callus_rib_fracture'), np.str_('hemidiaphragm_elevation'), np.str_('vascular_hilar_enlargement'), np.str_('aortic_elongation'), np.str_('calcified_granuloma'), np.str_('scoliosis'), np.str_('aortic_atheromatosis'), np.str_('infiltrates'), np.str_('diaphragmatic_eventration')]
  Val   clases (5): ['laminar_atelectasis', 'fibrotic_band', 'interstitial_pattern', 'costophrenic_angle_blunting', 'hiatal_hernia']
[Few-shot config] N_WAY=5 | N_SHOT=20 | N_QUERY=10


Epoch 1/15: 100%|██████████| 100/100 [11:17<00:00,  6.78s/it, acc=0.321, f1=0.293, invT=3.03, loss=1.554, rec=0.321]


[Epoch 1/15] loss=1.5535 train acc=0.3212 rec=0.3212 f1=0.2927 | val acc=0.2648 rec=0.2648 f1=0.2481


Epoch 2/15: 100%|██████████| 100/100 [00:28<00:00,  3.47it/s, acc=0.345, f1=0.311, invT=3.06, loss=1.520, rec=0.345]


[Epoch 2/15] loss=1.5196 train acc=0.3446 rec=0.3446 f1=0.3114 | val acc=0.2694 rec=0.2694 f1=0.2463


...

[Epoch 14/15] loss=1.4014 train acc=0.4566 rec=0.4566 f1=0.4267 | val acc=0.3442 rec=0.3442 f1=0.3156


Epoch 15/15: 100%|██████████| 100/100 [00:29<00:00,  3.38it/s, acc=0.446, f1=0.415, invT=3.28, loss=1.413, rec=0.446]


[Epoch 15/15] loss=1.4129 train acc=0.4462 rec=0.4462 f1=0.4151 | val acc=0.3475 rec=0.3475 f1=0.3192
  → E:/TFM/Nuevos_modelos/Outputs_matching_resnet18_5way_20shot\cv_fold_4\training_curves.png



[val]
  Loss episodio media: 1.5082
  Acc episodio media : 0.3432
  Recall episodio    : 0.3432
  F1 episodio        : 0.3159
  Acc global         : 0.3432
  F1 macro           : 0.3430
  ROC AUC macro OVR  : 0.6682



[test]
  Loss episodio media: 1.5174
  Acc episodio media : 0.3415
  Recall episodio    : 0.3415
  F1 episodio        : 0.3241
  Acc global         : 0.3415
  F1 macro           : 0.3414
  ROC AUC macro OVR  : 0.6654

=== CV SUMMARY ===
  [VAL] acc=0.3166±0.0362  rec=0.3166±0.0362  f1=0.2931±0.0349  roc=0.6386±0.0391
  [TEST] acc=0.3526±0.0125  rec=0.3526±0.0125  f1=0.3331±0.0100  roc=0.6688±0.0082
[OK] Experimento añadido en fila 42 de E:/TFM/Nuevos_resultados.xlsx
     Test F1: 0.3331 ± 0.0100  |  Test AUC: 0.6688


## MobileViT v2

### Config

In [17]:
@dataclass
class CFG:
    DATA_ROOT     = "E:/TFM/Dataset_fewshot"
    OUT_DIR       = "E:/TFM/Nuevos_modelos/Outputs_matching_MobileViT_5way_20shot"
    BACKBONE_NAME = "MobileViTV2"
    IMG_SIZE      = 224
    MAXVAL        = 65535.0
    N_WAY    = 5
    N_SHOT   = 20
    N_QUERY  = 10
    TRAIN_EPISODES_PER_EPOCH = 100
    VAL_EPISODES             = 200
    EPOCHS       = 15
    LR           = 1e-4
    WEIGHT_DECAY = 1e-4
    TEMPERATURE_INIT_INV = 3.0
    FREEZE_ENCODER = True
    EMB_DIM  = 128
    USE_CV   = True
    N_FOLDS  = 4
    SEED     = 42
    NUM_WORKERS = 0 if platform.system().lower().startswith("win") else 4
    DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

cfg    = CFG()
MAXVAL = cfg.MAXVAL
os.makedirs(cfg.OUT_DIR, exist_ok=True)
device = torch.device(cfg.DEVICE)
print("Device:", device)

base_preproc_xrv = T.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(cfg.IMG_SIZE),
])
train_transform = TorchAugment()
eval_transform  = IdentityTransform()

Device: cuda


### Encoder

In [18]:
class MobileViTV2ProtoEncoder(nn.Module):
    """MobileViTV2-1.0 (ImageNet pretrained, timm) + projection head."""
    def __init__(self, emb_dim: int = 128, freeze: bool = True):
        super().__init__()
        backbone = timm.create_model('mobilevitv2_100', pretrained=True, num_classes=0)
        self.backbone = backbone  # feat_dim = 512
        self.proj = nn.Linear(512, emb_dim)
        self.emb_dim = emb_dim
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        feats = self.backbone(x)  # [B, 512]
        return F.normalize(self.proj(feats), dim=1)

### Run

In [19]:
def build_mobilevit():
    return MobileViTV2ProtoEncoder(emb_dim=cfg.EMB_DIM, freeze=cfg.FREEZE_ENCODER)

run(build_mobilevit, base_preproc=base_preproc_xrv)

Train: 2000 imgs, 20 clases
Test:  435  imgs, 5  clases
[CV] 20 clases meta_train | 4 folds de clases

===== Fold 1/4 =====
  Train clases (15): [np.str_('gynecomastia'), np.str_('cardiomegaly'), np.str_('vertebral_anterior_compression'), np.str_('apical_pleural_thickening'), np.str_('alveolar_pattern'), np.str_('nodule'), np.str_('callus_rib_fracture'), np.str_('hemidiaphragm_elevation'), np.str_('vascular_hilar_enlargement'), np.str_('aortic_elongation'), np.str_('calcified_granuloma'), np.str_('scoliosis'), np.str_('aortic_atheromatosis'), np.str_('infiltrates'), np.str_('diaphragmatic_eventration')]
  Val   clases (5): ['laminar_atelectasis', 'fibrotic_band', 'interstitial_pattern', 'costophrenic_angle_blunting', 'hiatal_hernia']
[Few-shot config] N_WAY=5 | N_SHOT=20 | N_QUERY=10


Epoch 1/15: 100%|██████████| 100/100 [11:26<00:00,  6.87s/it, acc=0.310, f1=0.295, invT=3.03, loss=1.564, rec=0.310]


[Epoch 1/15] loss=1.5640 train acc=0.3098 rec=0.3098 f1=0.2949 | val acc=0.2852 rec=0.2852 f1=0.2558


Epoch 2/15: 100%|██████████| 100/100 [00:53<00:00,  1.86it/s, acc=0.327, f1=0.301, invT=3.06, loss=1.535, rec=0.327]


[Epoch 2/15] loss=1.5350 train acc=0.3272 rec=0.3272 f1=0.3012 | val acc=0.2822 rec=0.2822 f1=0.2410


...

[Epoch 14/15] loss=1.4351 train acc=0.4246 rec=0.4246 f1=0.3906 | val acc=0.3097 rec=0.3097 f1=0.2862


Epoch 15/15: 100%|██████████| 100/100 [00:45<00:00,  2.19it/s, acc=0.417, f1=0.387, invT=3.27, loss=1.441, rec=0.417]


[Epoch 15/15] loss=1.4414 train acc=0.4172 rec=0.4172 f1=0.3869 | val acc=0.3050 rec=0.3050 f1=0.2827
  → E:/TFM/Nuevos_modelos/Outputs_matching_MobileViT_5way_20shot\cv_fold_4\training_curves.png



[val]
  Loss episodio media: 1.5289
  Acc episodio media : 0.2989
  Recall episodio    : 0.2989
  F1 episodio        : 0.2750
  Acc global         : 0.2989
  F1 macro           : 0.2988
  ROC AUC macro OVR  : 0.6443



[test]
  Loss episodio media: 1.5093
  Acc episodio media : 0.3603
  Recall episodio    : 0.3603
  F1 episodio        : 0.3375
  Acc global         : 0.3603
  F1 macro           : 0.3601
  ROC AUC macro OVR  : 0.6748

=== CV SUMMARY ===
  [VAL] acc=0.3036±0.0318  rec=0.3036±0.0318  f1=0.2799±0.0316  roc=0.6312±0.0434
  [TEST] acc=0.3679±0.0077  rec=0.3679±0.0077  f1=0.3467±0.0070  roc=0.6787±0.0096
[OK] Experimento añadido en fila 43 de E:/TFM/Nuevos_resultados.xlsx
     Test F1: 0.3467 ± 0.0070  |  Test AUC: 0.6787


## Medical Mae

### Imports

In [13]:
sys.path.insert(0, "E:/TFM/Scripts")
import models_vit

### Config

In [14]:
@dataclass
class CFG:
    # Paths
    DATA_ROOT     = "E:/TFM/Dataset_fewshot"
    OUT_DIR       = "E:/TFM/Nuevos_modelos/Outputs_matching_medicalmae_5way_20shot"
    BACKBONE_NAME = "Medical_MAE_ViT-S"
    # Preproc
    IMG_SIZE   = 224
    MAXVAL     = 65535.0
    # Few-shot
    N_WAY   = 5
    N_SHOT  = 20
    N_QUERY = 10
    TRAIN_EPISODES_PER_EPOCH = 100
    VAL_EPISODES            = 200
    # Training
    EPOCHS              = 15
    LR                  = 1e-4
    WEIGHT_DECAY        = 1e-4
    TEMPERATURE_INIT_INV = 3.0
    SCHEDULER = "cosine"
    # Encoder
    XRV_WEIGHTS    = "densenet121-res224-all"
    FREEZE_ENCODER = True
    EMB_DIM        = 128
    # CV
    USE_CV                = True
    N_FOLDS               = 4
    CV_OVER_TRAIN_PLUS_VAL = False
    # System
    SEED        = 42
    NUM_WORKERS = 0 if platform.system().lower().startswith("win") else 4
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

cfg    = CFG()
MAXVAL = cfg.MAXVAL
os.makedirs(cfg.OUT_DIR, exist_ok=True)
device = torch.device(cfg.DEVICE)
print("Device:", device)

base_preproc_xrv = T.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(cfg.IMG_SIZE),
])
train_transform = TorchAugment()
eval_transform  = IdentityTransform()


Device: cuda


### Encoder

In [15]:
class MedicalMAEProtoEncoder(nn.Module):
    def __init__(self, emb_dim: int = 128, freeze: bool = True):
        super().__init__()
        import timm
        vit = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
        
        ckpt = torch.load("E:/TFM/Scripts/vit-s_CXR_0.3M_mae.pth", map_location="cpu")
        state_dict = ckpt['model'] if 'model' in ckpt else ckpt
        
        # Filtrar solo keys del encoder
        encoder_keys = {k: v for k, v in state_dict.items() 
                       if not k.startswith('decoder') and 
                          not k.startswith('mask_token') and
                          k != 'decoder_pos_embed'}
        
        msg = vit.load_state_dict(encoder_keys, strict=False)
        print("Medical_MAE load:", msg)
        
        self.backbone = vit
        self.proj = nn.Linear(384, emb_dim)
        self.emb_dim = emb_dim
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        feats = self.backbone(x)  # [B, 384]
        return F.normalize(self.proj(feats), dim=1)

### Run

In [16]:
def build_medical_mae():
    return MedicalMAEProtoEncoder(emb_dim=cfg.EMB_DIM, freeze=cfg.FREEZE_ENCODER)

run(build_medical_mae, base_preproc=base_preproc_xrv)

Train: 2000 imgs, 20 clases
Test:  435  imgs, 5  clases
[CV] 20 clases meta_train | 4 folds de clases

===== Fold 1/4 =====
  Train clases (15): [np.str_('gynecomastia'), np.str_('cardiomegaly'), np.str_('vertebral_anterior_compression'), np.str_('apical_pleural_thickening'), np.str_('alveolar_pattern'), np.str_('nodule'), np.str_('callus_rib_fracture'), np.str_('hemidiaphragm_elevation'), np.str_('vascular_hilar_enlargement'), np.str_('aortic_elongation'), np.str_('calcified_granuloma'), np.str_('scoliosis'), np.str_('aortic_atheromatosis'), np.str_('infiltrates'), np.str_('diaphragmatic_eventration')]
  Val   clases (5): ['laminar_atelectasis', 'fibrotic_band', 'interstitial_pattern', 'costophrenic_angle_blunting', 'hiatal_hernia']
[Few-shot config] N_WAY=5 | N_SHOT=20 | N_QUERY=10
Medical_MAE load: <All keys matched successfully>


Epoch 1/15: 100%|██████████| 100/100 [06:20<00:00,  3.80s/it, acc=0.267, f1=0.218, invT=3.02, loss=1.579, rec=0.267]


[Epoch 1/15] loss=1.5791 train acc=0.2674 rec=0.2674 f1=0.2184 | val acc=0.2342 rec=0.2342 f1=0.1980


Epoch 2/15: 100%|██████████| 100/100 [00:40<00:00,  2.45it/s, acc=0.264, f1=0.217, invT=3.03, loss=1.568, rec=0.264]


[Epoch 2/15] loss=1.5676 train acc=0.2638 rec=0.2638 f1=0.2170 | val acc=0.2374 rec=0.2374 f1=0.1856


...

[Epoch 14/15] loss=1.5561 train acc=0.2864 rec=0.2864 f1=0.2350 | val acc=0.2628 rec=0.2628 f1=0.2096


Epoch 15/15: 100%|██████████| 100/100 [00:41<00:00,  2.43it/s, acc=0.289, f1=0.237, invT=3.06, loss=1.555, rec=0.289]


[Epoch 15/15] loss=1.5545 train acc=0.2888 rec=0.2888 f1=0.2372 | val acc=0.2628 rec=0.2628 f1=0.2121
  → E:/TFM/Nuevos_modelos/Outputs_matching_medicalmae_5way_20shot\cv_fold_4\training_curves.png



[val]
  Loss episodio media: 1.5651
  Acc episodio media : 0.2623
  Recall episodio    : 0.2623
  F1 episodio        : 0.2079
  Acc global         : 0.2623
  F1 macro           : 0.2623
  ROC AUC macro OVR  : 0.5993



[test]
  Loss episodio media: 1.5750
  Acc episodio media : 0.2758
  Recall episodio    : 0.2758
  F1 episodio        : 0.2375
  Acc global         : 0.2758
  F1 macro           : 0.2756
  ROC AUC macro OVR  : 0.5937

=== CV SUMMARY ===
  [VAL] acc=0.2742±0.0448  rec=0.2742±0.0448  f1=0.2221±0.0364  roc=0.5999±0.0413
  [TEST] acc=0.2783±0.0054  rec=0.2783±0.0054  f1=0.2361±0.0072  roc=0.5963±0.0034
[OK] Experimento añadido en fila 45 de E:/TFM/Nuevos_resultados.xlsx
     Test F1: 0.2361 ± 0.0072  |  Test AUC: 0.5963
